In [10]:
# Block 1: Imports
import os
import cv2
import torch
import numpy as np
import pandas as pd
import ast
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO, RTDETR
from collections import deque, Counter
import logging
import pkg_resources
# Keep Ultralytics quiet in notebook output
logging.getLogger("ultralytics").setLevel(logging.ERROR)


In [11]:
# Block 2: Workspace and lightweight video validation for direct inference/tracking

def initialize_workspace():
    base = Path.cwd().absolute()
    paths = {
        "videos": base / "raw_data" / "videos",
        "tracks": base / "Runs" / "trajectories",
        "renders": base / "Runs" / "annotated_videos",
        "models": base / "models",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def open_video_capture(video_path):
    video_path = Path(video_path)

    backends = [cv2.CAP_MSMF, cv2.CAP_DSHOW, cv2.CAP_ANY] if os.name == "nt" else [cv2.CAP_ANY]

    for api in backends:
        cap = cv2.VideoCapture(str(video_path), api)
        if cap.isOpened():
            ret, _ = cap.read()
            if ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                return cap
            cap.release()

    return None


def get_video_metadata(video_path):
    cap = open_video_capture(video_path)
    if cap is None:
        return None

    meta = {
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "total": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    return meta


def validate_video_input(video_path):
    video_path = Path(video_path)

    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    cap = open_video_capture(video_path)
    if cap is None:
        return False

    cap.release()
    return True

In [12]:
# Block 3: Group class and Tracker
def get_grouped_class(source_label, car_group):
    # STRICT MAPPING: Every possible detection is forced into one of the 7 official classes
    mapping = {
        "CAR_GROUP": "Car", "Car": "Car", "Hatchback": "Car", "Sedan": "Car", "SUV": "Car", "MUV": "Car", "Van": "Car",
        "Truck": "HCV", "HCV": "HCV", "Tractor": "HCV",
        "Auto Rickshaw": "Three-wheeler", "Rickshaw": "Three-wheeler", "Three-wheeler": "Three-wheeler",
        "Motorcycle": "Two-wheeler", "Bicycle": "Two-wheeler", "Two-wheeler": "Two-wheeler",
        "Mini Bus": "Bus", "Bus": "Bus",
        "Tempo": "LCV", "LCV": "LCV",
        "person": "Pedestrian", "Pedestrian": "Pedestrian"
    }
    # Return the mapped name, or "Car" as a safe fallback if for some reason the name is unknown
    return mapping.get(source_label, "Car")


def get_ultra_tracker_yaml(mode):
    if mode.endswith(".yaml"): return mode
    if mode == "botsort": return "botsort.yaml"
    if mode == "bytetrack": return "bytetrack.yaml"
    raise ValueError(f"Unsupported tracker_mode: {mode}")


In [13]:
# Block 4: One-pass trajectory extraction and annotated rendering

def extract_trajectories(model, v_path, c_path, out_path, meta, cfg, ped_model=None):
    cap = open_video_capture(v_path)
    if cap is None: raise ValueError(f"Could not open video: {v_path}")
    cap.set(cv2.CAP_PROP_POS_FRAMES, cfg["start_frame"])
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), meta["fps"], (meta["width"], meta["height"]))
    data_log = []
    ultra_yaml = get_ultra_tracker_yaml(cfg["tracker_mode"])
    frame_idx = cfg["start_frame"]
    total_to_read = max(0, cfg["end_frame"] - cfg["start_frame"])

    for _ in tqdm(range(total_to_read), desc="Task 3-4: Track + Render"):
        ret, frame = cap.read()
        if not ret: break
        all_boxes = []
        
        # 1) Main Vehicle Tracking
        v_res = model.track(frame, persist=True, tracker=ultra_yaml, conf=cfg["conf"], iou=cfg["iou"], imgsz=1080, verbose=False)
        if v_res[0].boxes is not None and v_res[0].boxes.id is not None:
            for box, tid, c_idx, score in zip(v_res[0].boxes.xyxy.cpu().numpy(), 
                                            v_res[0].boxes.id.cpu().numpy().astype(int), 
                                            v_res[0].boxes.cls.cpu().numpy().astype(int), 
                                            v_res[0].boxes.conf.cpu().numpy()):
                src_label = model.names[c_idx]
                grp_label = get_grouped_class(src_label, cfg["car_group"])
                all_boxes.append((box, tid, grp_label, src_label, score))

        # 2) Pedestrian Tracking
        if ped_model is not None:
            p_res = ped_model.track(frame, persist=True, classes=[0], conf=0.25, verbose=False)
            if p_res[0].boxes is not None and p_res[0].boxes.id is not None:
                for box, tid, c_idx, score in zip(p_res[0].boxes.xyxy.cpu().numpy(), 
                                                p_res[0].boxes.id.cpu().numpy().astype(int), 
                                                p_res[0].boxes.cls.cpu().numpy().astype(int), 
                                                p_res[0].boxes.conf.cpu().numpy()):
                    all_boxes.append((box, tid + 10000, "Pedestrian", "person", score))

        ann_frame = frame.copy()
        for box, tid, grp_label, src_label, score in all_boxes:
            x1, y1, x2, y2 = [int(v) for v in box]
            data_log.append({
                "frame": frame_idx, "track_id": tid, "grouped_class": grp_label,
                "source_class": src_label, "confidence": float(score),
                "bbox": [round(float(v), 2) for v in box]
            })
            cv2.rectangle(ann_frame, (x1, y1), (x2, y2), (0, 255, 0), cfg["line_thickness"])
            cv2.putText(ann_frame, f"ID:{tid} {grp_label}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        writer.write(ann_frame)
        frame_idx += 1

    cap.release(); writer.release()
    pd.DataFrame(data_log).to_csv(c_path, index=False)


In [14]:
# Block 5: Annotated video features
def build_display_color_map(model_names, car_group):
    """
    Build a stable, explicit colour map.
    CAR_GROUP is forced to green.
    Common non-group labels get fixed colours.
    Any remaining labels get colours from a fallback palette that does not use green.
    """
    if isinstance(model_names, dict):
        labels = list(model_names.values())
    else:
        labels = list(model_names)

    # Keep original order but remove duplicates
    ordered_labels = []
    for lab in labels:
        lab = str(lab)
        if lab not in ordered_labels:
            ordered_labels.append(lab)

    color_map = {
        "CAR_GROUP": (0, 255, 0),   # green, fixed
        "LCV": (255, 0, 0),         # blue
        "Bus": (0, 165, 255),       # orange
        "Mini Bus": (255, 0, 255),  # magenta
        "Truck": (0, 0, 255),       # red
        "Auto Rickshaw": (255, 255, 0),
        "Rickshaw": (0, 255, 255),
        "Bicycle": (255, 128, 0),
        "Motorcycle": (128, 0, 255),
        "Tempo": (128, 128, 0),
    }

    fallback_palette = [
        (255, 128, 0),
        (128, 0, 255),
        (0, 255, 255),
        (255, 0, 255),
        (0, 0, 255),
        (0, 165, 255),
        (255, 255, 0),
        (128, 128, 0),
        (0, 128, 255),
        (255, 64, 64),
        (64, 255, 64),
        (64, 64, 255),
        (200, 100, 0),
        (100, 0, 200),
    ]

    used = set(color_map.values())
    fallback_idx = 0

    for label in ordered_labels:
        if label in car_group:
            color_map[label] = (0, 255, 0)
            continue

        if label in color_map:
            continue

        # Pick the next unused fallback colour
        while fallback_idx < len(fallback_palette) and fallback_palette[fallback_idx] in used:
            fallback_idx += 1

        if fallback_idx < len(fallback_palette):
            color_map[label] = fallback_palette[fallback_idx]
            used.add(fallback_palette[fallback_idx])
            fallback_idx += 1
        else:
            # Last-resort distinct colour, still not green
            color_map[label] = (180, 180, 60)

    return color_map




In [15]:
# Block 6: CSV and XML of bboxes
def parse_bbox(value):
    if pd.isna(value):
        return None

    if isinstance(value, (list, tuple)):
        if len(value) != 4:
            return None
        return [float(v) for v in value]

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
        except Exception:
            return None

        if isinstance(parsed, (list, tuple)) and len(parsed) == 4:
            return [float(v) for v in parsed]

    return None


def validate_and_prepare_csv(csv_path):
    df = pd.read_csv(csv_path)

    required_cols = ["frame", "track_id", "grouped_class", "bbox"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in CSV: {missing_cols}")

    df = df.copy()
    df = df[df["grouped_class"].notna()].copy()

    if df.empty:
        raise ValueError("CSV contains no valid grouped_class rows.")

    df["frame"] = pd.to_numeric(df["frame"], errors="raise").astype(int)
    df["track_id"] = pd.to_numeric(df["track_id"], errors="raise").astype(int)
    df["grouped_class"] = df["grouped_class"].astype(str)

    parsed_bboxes = df["bbox"].apply(parse_bbox)

    if parsed_bboxes.isna().any():
        bad_rows = df.loc[parsed_bboxes.isna(), ["frame", "track_id", "bbox"]]
        raise ValueError(
            "Some bbox values could not be parsed.\n"
            f"{bad_rows.head(10).to_string(index=False)}"
        )

    bbox_df = pd.DataFrame(
        parsed_bboxes.tolist(),
        columns=["xtl", "ytl", "xbr", "ybr"],
        index=df.index,
    )
    df = pd.concat([df, bbox_df], axis=1)

    invalid = df[(df["xbr"] <= df["xtl"]) | (df["ybr"] <= df["ytl"])]
    if not invalid.empty:
        raise ValueError(
            "Invalid bbox geometry found.\n"
            f"{invalid[['frame', 'track_id', 'bbox']].head(10).to_string(index=False)}"
        )

    return df


def build_cvat_video_xml(df, xml_out_path, task_name, task_size, video_width, video_height):
    xml_out_path = Path(xml_out_path)
    xml_out_path.parent.mkdir(parents=True, exist_ok=True)

    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"

    meta = ET.SubElement(root, "meta")
    task = ET.SubElement(meta, "task")
    ET.SubElement(task, "id").text = "0"
    ET.SubElement(task, "name").text = str(task_name)
    ET.SubElement(task, "size").text = str(int(task_size))
    ET.SubElement(task, "mode").text = "interpolation"
    ET.SubElement(task, "overlap").text = "0"
    ET.SubElement(task, "bugtracker").text = ""
    ET.SubElement(task, "flipped").text = "False"
    ET.SubElement(task, "created").text = ""
    ET.SubElement(task, "updated").text = ""

    labels_el = ET.SubElement(task, "labels")
    label_names = sorted(df["grouped_class"].unique().tolist())

    for label_name in label_names:
        label_el = ET.SubElement(labels_el, "label")
        ET.SubElement(label_el, "name").text = label_name
        ET.SubElement(label_el, "type").text = "bbox"
        ET.SubElement(label_el, "attributes")

    segments = ET.SubElement(task, "segments")
    segment = ET.SubElement(segments, "segment")
    ET.SubElement(segment, "id").text = "0"
    ET.SubElement(segment, "start").text = "0"
    ET.SubElement(segment, "stop").text = str(int(task_size) - 1)
    ET.SubElement(segment, "url").text = ""

    owner = ET.SubElement(task, "owner")
    ET.SubElement(owner, "username").text = ""
    ET.SubElement(owner, "email").text = ""

    original_size = ET.SubElement(task, "original_size")
    ET.SubElement(original_size, "width").text = str(int(video_width))
    ET.SubElement(original_size, "height").text = str(int(video_height))

    ET.SubElement(meta, "dumped").text = pd.Timestamp.utcnow().strftime(
        "%Y-%m-%d %H:%M:%S.%f+00:00"
    )

    df = df.sort_values(["track_id", "frame"]).reset_index(drop=True)
    grouped = df.groupby("track_id", sort=True)

    for track_id, track_df in tqdm(
        grouped,
        total=df["track_id"].nunique(),
        desc="Task 5: CVAT XML"
    ):
        label_counts = Counter(track_df["grouped_class"].tolist())
        track_label = label_counts.most_common(1)[0][0]

        track_el = ET.SubElement(root, "track", id=str(int(track_id)), label=track_label, source="auto")
        frames = track_df["frame"].tolist()

        # Visible boxes
        for row in track_df.itertuples(index=False):
            ET.SubElement(
                track_el,
                "box",
                frame=str(int(row.frame)),
                xtl=f"{float(row.xtl):.2f}",
                ytl=f"{float(row.ytl):.2f}",
                xbr=f"{float(row.xbr):.2f}",
                ybr=f"{float(row.ybr):.2f}",
                outside="0",
                occluded="0",
                keyframe="1",
            )

        # Track termination
        last_frame = frames[-1]
        ET.SubElement(
            track_el,
            "box",
            frame=str(int(last_frame) + 1),
            xtl=f"{float(track_df.iloc[-1].xtl):.2f}",
            ytl=f"{float(track_df.iloc[-1].ytl):.2f}",
            xbr=f"{float(track_df.iloc[-1].xbr):.2f}",
            ybr=f"{float(track_df.iloc[-1].ybr):.2f}",
            outside="1",
            occluded="0",
            keyframe="1",
        )

    try:
        ET.indent(root, space="  ", level=0)
    except AttributeError:
        pass

    tree = ET.ElementTree(root)
    tree.write(xml_out_path, encoding="utf-8", xml_declaration=True)

    print(f"Success: CVAT XML saved to {xml_out_path}")
    print(f"Tracks written: {df['track_id'].nunique()}")

In [16]:
# Block 7: Validation
def validate_cvat_xml(xml_path):
    """
    Load a CVAT Video 1.1 XML file and perform basic structural checks.
    This does not import into CVAT; it only verifies that the file is well-formed
    and contains the expected annotation elements.
    """
    xml_path = Path(xml_path)

    if not xml_path.exists():
        raise FileNotFoundError(f"XML file not found: {xml_path}")

    tree = ET.parse(xml_path)
    root = tree.getroot()

    if root.tag != "annotations":
        raise ValueError(f"Unexpected root tag: {root.tag}. Expected 'annotations'.")

    version_el = root.find("version")
    if version_el is None:
        raise ValueError("Missing <version> element in XML.")
    print(f"XML version: {version_el.text}")

    meta = root.find("meta")
    if meta is None:
        raise ValueError("Missing <meta> section in XML.")

    task = meta.find("task")
    if task is None:
        raise ValueError("Missing <task> section inside <meta>.")

    name_el = task.find("name")
    mode_el = task.find("mode")
    size_el = task.find("size")

    print(f"Task name: {name_el.text if name_el is not None else 'N/A'}")
    print(f"Task mode: {mode_el.text if mode_el is not None else 'N/A'}")
    print(f"Task size: {size_el.text if size_el is not None else 'N/A'}")

    labels = task.find("labels")
    if labels is None:
        raise ValueError("Missing <labels> section inside <task>.")

    label_names = []
    for label in labels.findall("label"):
        name = label.find("name")
        if name is not None and name.text:
            label_names.append(name.text)

    if not label_names:
        raise ValueError("No labels found in XML.")

    print(f"Labels found ({len(label_names)}): {label_names}")

    tracks = root.findall("track")
    if not tracks:
        raise ValueError("No <track> elements found in XML.")

    print(f"Tracks found: {len(tracks)}")

    print("\nSample track inspection:")
    for track in tracks[:3]:
        track_id = track.attrib.get("id", "N/A")
        track_label = track.attrib.get("label", "N/A")
        boxes = track.findall("box")
        print(f"  Track id={track_id}, label={track_label}, boxes={len(boxes)}")

        for box in boxes[:2]:
            frame = box.attrib.get("frame", "N/A")
            xtl = box.attrib.get("xtl", "N/A")
            ytl = box.attrib.get("ytl", "N/A")
            xbr = box.attrib.get("xbr", "N/A")
            ybr = box.attrib.get("ybr", "N/A")
            outside = box.attrib.get("outside", "N/A")
            keyframe = box.attrib.get("keyframe", "N/A")
            print(
                f"    box frame={frame}, xtl={xtl}, ytl={ytl}, "
                f"xbr={xbr}, ybr={ybr}, outside={outside}, keyframe={keyframe}"
            )

    print(f"\nValidation passed: {xml_path}")

In [17]:
# Block 8: Tracking
def print_track_diagnostics(csv_path):
    df = pd.read_csv(csv_path)

    track_sizes = df.groupby("track_id").size().sort_values()

    print("Total rows:", len(df))
    print("Unique track_ids:", df["track_id"].nunique())
    print("\nTrack length summary:")
    print(track_sizes.describe())

    print("\nSingle-frame tracks:", int((track_sizes == 1).sum()))
    print("Tracks with 2-3 frames:", int(((track_sizes >= 2) & (track_sizes <= 3)).sum()))
    print("Tracks with >50 frames:", int((track_sizes > 50).sum()))

    duplicate_pairs = df.duplicated(["frame", "track_id"], keep=False).sum()
    print("\nDuplicate (frame, track_id) rows:", int(duplicate_pairs))

    print("\nSmallest tracks:")
    print(track_sizes.head(20))

    print("\nLargest tracks:")
    print(track_sizes.tail(20))

In [18]:
# Block 9: Workspace setup, config, and execution (GPU Optimized)
import torch
PROJ_PATHS = initialize_workspace()
CONFIG = {
    "start_frame": 0, "end_frame": 23262, "frame_stride": 1, "conf": 0.45, "iou": 0.2,
    "tracker_mode": "dense_traffic_tracker.yaml", "line_thickness": 2, "trail_duration": 2.0,
    "car_group": {"Hatchback", "Sedan", "SUV", "MUV", "Van"}
}
VIDEO_IN = PROJ_PATHS["videos"] / "76_RC_PTZ 1_20260427_210000_20260427_211500_4.mp4"
VIDEO_OUT = PROJ_PATHS["renders"] / "annotated_junction24.mp4"
CSV_OUT = PROJ_PATHS["tracks"] / "trajectories24.csv"
XML_OUT = PROJ_PATHS["tracks"] / "cvat_annotations24.xml"
MODEL_PATH = PROJ_PATHS["models"] / "UVH-26-MV-YOLOv11-S.pt"

video_meta = get_video_metadata(VIDEO_IN)
# Force GPU usage with .to("cuda")
device = "cuda" if torch.cuda.is_available() else "cpu"
traffic_model = YOLO(str(MODEL_PATH)).to(device)
ped_model = YOLO("yolo11n.pt").to(device) # Switched to Nano for 10x speedup

extract_trajectories(traffic_model, VIDEO_IN, CSV_OUT, VIDEO_OUT, video_meta, CONFIG, ped_model=ped_model)

prepared_df = validate_and_prepare_csv(CSV_OUT)
build_cvat_video_xml(df=prepared_df, xml_out_path=XML_OUT, task_name=VIDEO_IN.stem, task_size=video_meta["total"], video_width=video_meta["width"], video_height=video_meta["height"])
print(f"--- Success (Using {device}) ---")


Task 5: CVAT XML: 100%|██████████| 4280/4280 [00:04<00:00, 867.78it/s] 


Success: CVAT XML saved to /home/manav/My-Space/G-TRISP/Annotation/Runs/trajectories/cvat_annotations24.xml
Tracks written: 4280
--- Success (Using cuda) ---
